<a href="https://colab.research.google.com/github/juanatga/dsb_capstone/blob/main/Missing_data_in_Time_Series.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook is to clarify the question about how to detect and manage missing values in the index and rows of a time-series dataframe WITH PANDAS

# Generating synthetic data to discuss

In [ ]:
# Generate data with gaps

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

N_ROWS = 100

# --- Reference pools ---------------------------------------------------
stores = ["North", "South", "East", "West", "Central"]
customers = [
    "Alice Moreau", "Bruno Klein", "Chloé Dupont", "Daniel Rossi", "Elena Novak",
    "Farid Haddad", "Greta Lindqvist", "Hugo Peeters", "Ines Ferreira", "Jonas Weber",
    "Katrin Vogel", "Luca Bianchi", "Marta Silva", "Niels Bakker", "Olivia Grant",
    "Pavel Horák", "Quentin Roux", "Rosa Marín", "Stefan Popa", "Teresa Costa",
]

# --- Dates with deliberate gaps ---------------------------------------
# Start 01 Jan 2026, sample 240 calendar days but keep only ~100 of them.
start = pd.Timestamp("2026-01-01")
candidate_days = pd.date_range(start, periods=240, freq="D")

# Keep roughly 60% of days -> uneven coverage, plus natural weekend clustering
keep_mask = rng.random(len(candidate_days)) < 0.62
transaction_days = candidate_days[keep_mask]

# Sample exactly N_ROWS dates WITH replacement so some days have several
# transactions and others none at all.
dates = pd.Series(rng.choice(transaction_days, size=N_ROWS)).sort_values().reset_index(drop=True)

# --- Numeric columns with missing values ------------------------------
items = rng.integers(1, 25, size=N_ROWS).astype(float)          # 1–24 items
unit_price = rng.uniform(8, 120, size=N_ROWS)                    # €8–€120 per item
total_value = items * unit_price

# Inject missing values (~12% in each numeric column, independently)
items_missing = rng.random(N_ROWS) < 0.12
value_missing = rng.random(N_ROWS) < 0.12

items[items_missing] = np.nan
total_value[value_missing] = np.nan

df = pd.DataFrame({
    "transaction_date": dates,
    "total_value_eur": np.round(total_value, 2),
    "num_items": items,
    "store_name": rng.choice(stores, size=N_ROWS),
    "customer_name": rng.choice(customers, size=N_ROWS),
})

print(df.head(10))
print("\nShape:", df.shape)
print("\nMissing values per column:\n", df.isna().sum())
print("\nUnique transaction days:", df["transaction_date"].nunique(),
      "out of a", (df["transaction_date"].max() - df["transaction_date"].min()).days + 1,
      "day span")

  transaction_date  total_value_eur  num_items store_name    customer_name
0       2026-01-05            70.17        3.0       West     Teresa Costa
1       2026-01-10          2423.29       21.0       West     Farid Haddad
2       2026-01-16           293.10        3.0      South  Greta Lindqvist
3       2026-01-17           148.98        2.0    Central       Rosa Marín
4       2026-01-17           956.54       10.0      South     Hugo Peeters
5       2026-01-18          2232.22       23.0      North      Elena Novak
6       2026-01-22          1937.24       17.0    Central     Luca Bianchi
7       2026-02-02              NaN        3.0      South  Greta Lindqvist
8       2026-02-03          1555.86        NaN      North       Rosa Marín
9       2026-02-04           391.56       21.0       West      Stefan Popa

Shape: (100, 5)

Missing values per column:
 transaction_date     0
total_value_eur     13
num_items           11
store_name           0
customer_name        0
dtype: int64



# How to check for gaps in the timestamps

In [ ]:
# First, check the data type of transaction date is datetime
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_date  100 non-null    datetime64[ns]
 1   total_value_eur   87 non-null     float64       
 2   num_items         89 non-null     float64       
 3   store_name        100 non-null    object        
 4   customer_name     100 non-null    object        
dtypes: datetime64[ns](1), float64(2), object(2)
memory usage: 4.0+ KB


In [ ]:
# I can calculate the difference between each date and the previous one (by default)
date_diffs = df.transaction_date.diff()

# This "column" is a series of Timedeltas, not integers ! That's important

display(date_diffs)

,transaction_date
0,NaT
1,5 days
2,6 days
3,1 days
4,0 days
...,...
95,3 days
96,2 days
97,1 days
98,6 days


In [ ]:
# It is easier if I see the value counts of that
date_diffs.value_counts(sort=True, normalize=False)

,count
transaction_date,
1 days,25
0 days,21
2 days,16
3 days,12
4 days,8
6 days,7
5 days,5
7 days,2
11 days,1


In [ ]:
# Let's say I want to see the rows where I have a gap of 5 days on the transaction date (no transactions for the past 5 days??!)

# The limit has to be a time delta too
limit = pd.Timedelta("5 days")

display(df[date_diffs == limit])

,transaction_date,total_value_eur,num_items,store_name,customer_name
1,2026-01-10,2423.29,21.0,West,Farid Haddad
12,2026-02-10,427.65,6.0,East,Teresa Costa
50,2026-05-04,1428.34,18.0,East,Jonas Weber
58,2026-05-16,1187.15,20.0,West,Greta Lindqvist
65,2026-05-30,851.76,8.0,Central,Alice Moreau


# What to do in practice?

In [ ]:
# Let's say I want to analyze total sales
df.set_index("transaction_date", inplace=True)
df.head()

,total_value_eur,num_items,store_name,customer_name
transaction_date,,,,
2026-01-05,70.17,3.0,West,Teresa Costa
2026-01-10,2423.29,21.0,West,Farid Haddad
2026-01-16,293.10,3.0,South,Greta Lindqvist
2026-01-17,148.98,2.0,Central,Rosa Marín
2026-01-17,956.54,10.0,South,Hugo Peeters


In [ ]:
# I can group per week or per month
display(df.resample("W").sum().info())
display(df.resample("W").sum().head())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 34 entries, 2026-01-11 to 2026-08-30
Freq: W-SUN
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   total_value_eur  34 non-null     float64
 1   num_items        34 non-null     float64
 2   store_name       34 non-null     object 
 3   customer_name    34 non-null     object 
dtypes: float64(2), object(2)
memory usage: 1.3+ KB


None

,total_value_eur,num_items,store_name,customer_name
transaction_date,,,,
2026-01-11,2493.46,24.0,WestWest,Teresa CostaFarid Haddad
2026-01-18,3630.84,38.0,SouthCentralSouthNorth,Greta LindqvistRosa MarínHugo PeetersElena Novak
2026-01-25,1937.24,17.0,Central,Luca Bianchi
2026-02-01,0.00,0.0,0,0
2026-02-08,4085.69,46.0,SouthNorthWestSouthCentral,Greta LindqvistRosa MarínStefan PopaGreta Lind...


# Specialized Python libraries for time series data cleaning and EDA

Found using the AI assitant https://lumo.proton.me



## **Core Time Series EDA & Cleaning Libraries**

### **tsfresh** — Automatic Feature Extraction
The most prominent library for automatically extracting features from time series data.

| Feature | Description |
|---------|-------------|
| **Purpose** | Extracts 1200+ statistical features from time series automatically |
| **Best for** | Converting raw time series into ML-ready feature matrices |
| **Key capabilities** | Imputation, feature selection, scalable to large datasets |

```python
from tsfresh import extract_features, extract_relevant_features
from tsfresh.utilities.dataframe_functions import impute

# Extract all features
extracted = extract_features(df, column_id="id", column_sort="time")

# Impute missing values created during extraction
impute(extracted)

# Extract only relevant features (single line!)
features = extract_relevant_features(df, target_y, column_id="id", column_sort="time")
```

---

### **sktime** — Unified Time Series Interface
Comprehensive toolkit with strong EDA and detection capabilities.

| Feature | Description |
|---------|-------------|
| **Purpose** | Unified scikit-learn-like interface for all time series tasks |
| **Best for** | Classification, forecasting, anomaly/changepoint detection |
| **Key capabilities** | Built-in detectors, visualizations, cross-validation |

```python
from sktime.detection import HampelDetector, SubLOF

# Anomaly detection
detector = HampelDetector(window_length=10, n_sigma=3)
anomalies = detector.fit_transform(series)

# Changepoint detection
from sktime.forecasting.compose import make_reduction
```

---

### **pmdarima** — Statistical Time Series Tools
Fills gaps in Python's time series analysis (equivalent to R's `auto.arima`).

| Feature | Description |
|---------|-------------|
| **Purpose** | Statistical testing, decomposition, transformations |
| **Best for** | Stationarity tests, seasonality analysis, Box-Cox transforms |
| **Key capabilities** | Auto-ARIMA, differencing, seasonal decomposition |

```python
import pmdarima as pm

# Stationarity tests
from pmdarima.arima.stationarity import adf_test
adf_result = adf_test(series)

# Seasonal decomposition
from pmdarima.utils import seasonal_decompose
decomposed = seasonal_decompose(series, model='additive', period=12)
decomposed.plot()
```

---

### **adtk** (Anomaly Detection Toolkit) — Rule-Based Anomaly Detection

| Feature | Description |
|---------|-------------|
| **Purpose** | Unsupervised, rule-based anomaly detection |
| **Best for** | Identifying outliers, spikes, drifts in time series |
| **Key capabilities** | Multiple detector types, visualization, pipeline composition |

```python
from adtk.visualization import visualize
from adtk.detector import SpikeDetector, LevelShiftDetector

# Detect spikes
spike_detector = SpikeDetector(c=3.0)
spikes = spike_detector.fit_detect(series)

# Combine multiple detectors
from adtk.pipeline import AnomalyPipeline
pipeline = AnomalyPipeline(
    transformer=None,
    detector=[spike_detector, level_shift_detector],
    aggregator='max'
)
```

---

### **darts** — Forecasting with Anomaly Detection

| Feature | Description |
|---------|-------------|
| **Purpose** | User-friendly forecasting and anomaly detection |
| **Best for** | Wrangling + forecasting + backtesting in one library |
| **Key capabilities** | Multi-model comparison, probabilistic forecasts |

```python
from darts import TimeSeries
from darts.models import ExponentialSmoothing, ARIMA

# Convert pandas to TimeSeries
ts = TimeSeries.from_dataframe(df, time_col='date', value_cols='value')

# Backtest easily
model = ARIMA()
forecast = model.fit(ts[:80]).predict(n=20)
```

---

## **Supporting Libraries**

| Library | Purpose |
|---------|---------|
| **Featuretools** | Entity-based feature extraction with time-aware primitives |
| **cesium** | Feature extraction for non-uniformly sampled signals |
| **catch22** | 22 canonical time series features (minimal, interpretable) |
| **tsfeatures** | Numpy/R-compatible time series feature library |

---

## **Recommended Workflow**

For efficient time series data cleaning and EDA:

1. **Initial inspection**: Use `pandas` + `seaborn` for quick plots
2. **Stationarity checks**: `pmdarima` for ADF/KPSS tests
3. **Anomaly detection**: `adtk` or `sktime.detection` for identifying issues
4. **Feature extraction**: `tsfresh` for ML-ready feature matrices
5. **Seasonal decomposition**: `pmdarima` or `statsmodels.tsa.seasonal_decompose`

```python
# Combined example
import pandas as pd
import pmdarima as pm
from tsfresh import extract_relevant_features
from adtk.detector import TrendDetector

# Load and initial cleanup
df = pd.read_csv('data.csv', parse_dates=['date'])

# Check stationarity
adf_stat, p_value = pm.arima.stationarity.adf_test(df['value'])

# Remove trends/anomalies
trend_detector = TrendDetector()
anomalous_idx = trend_detector.fit_detect(df['value'])
df_clean = df.drop(anomalous_idx.index)

# Extract features for modeling
features = extract_relevant_features(
    df_clean,
    target_column,
    column_id='id',
    column_sort='date'
)
```

---

## **Tradeoffs to Consider**

| Library | Pros | Cons |
|---------|------|------|
| **tsfresh** | Comprehensive, automated, well-tested | Slow on very large datasets; requires structured format |
| **sktime** | Complete ecosystem, consistent API | Steeper learning curve |
| **adtk** | Easy visualization, flexible detectors | Less automated than tsfresh |
| **pmdarima** | R-equivalent statistical power | Focused on forecasting, less on general EDA |